# Week 1: Data Collection and Preprocessing

**Dataset:** Global Superstore (Kaggle, apoorvaappz/global-super-store-dataset): 51,290 order lines, 2011–2014.

## 1. Data Loading, Cleaning and Feature Engineering

In [5]:
import numpy as np
import pandas as pd
from pathlib import Path

root = Path(r"C:\Users\tanis\OneDrive\Desktop\cs-vision-internship")
raw_files = [p for p in (root / "data").rglob("*.csv")
             if "superstore" in p.name.lower() and "clean" not in p.name.lower()]
assert raw_files, "No Superstore CSV found under the data folder"
RAW_PATH = raw_files[0]
CLEAN_DIR = root / "data" / "clean"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
print("Raw file:", RAW_PATH)

# 1. Load
raw = pd.read_csv(RAW_PATH, encoding="latin1")
df = raw.copy()
print("Raw shape:", df.shape)

# 2. Column names, duplicates, text
df.columns = df.columns.str.strip().str.replace(r"[\s\-]+", "_", regex=True)
df = df.drop_duplicates().reset_index(drop=True)
for c in df.select_dtypes(include=["object", "string"]).columns:
    df[c] = df[c].astype("string").str.strip()

# 3. Types and dates (file uses DD-MM-YYYY)
df["Order_Date"] = pd.to_datetime(df["Order_Date"], format="%d-%m-%Y")
df["Ship_Date"] = pd.to_datetime(df["Ship_Date"], format="%d-%m-%Y")
for c in ["Sales", "Profit", "Quantity", "Discount", "Shipping_Cost"]:
    df[c] = pd.to_numeric(df[c])

# 4. Missing values (Postal_Code is only filled for US orders)
df["Postal_Code"] = (df["Postal_Code"].astype("string")
                     .str.replace(r"\.0$", "", regex=True).fillna("Unknown"))

# 5. Derived finance fields
df["Cost"] = df.Sales - df.Profit
df["Gross_Profit"] = df.Profit
df["Operating_Expense"] = df.Shipping_Cost
df["Net_Profit"] = df.Gross_Profit - df.Operating_Expense
df["Gross_Margin_%"] = (df.Gross_Profit / df.Sales * 100).round(2)
df["Net_Margin_%"] = (df.Net_Profit / df.Sales * 100).round(2)
df["Cost_to_Revenue"] = (df.Cost / df.Sales).round(4)

# 6. Time fields
df["Year"] = df.Order_Date.dt.year
df["Quarter"] = df.Order_Date.dt.quarter
df["Month"] = df.Order_Date.dt.month
df["Month_Name"] = df.Order_Date.dt.strftime("%b")
df["Year_Month"] = df.Order_Date.dt.to_period("M").astype(str)
df["Day_of_Week"] = df.Order_Date.dt.day_name()
df["Shipping_Days"] = (df.Ship_Date - df.Order_Date).dt.days

# 7. Outlier flags (flagged, not removed)
def iqr_flag(s, k=3):
    q1, q3 = s.quantile([.25, .75]); i = q3 - q1
    return (s < q1 - k * i) | (s > q3 + k * i)
df["Sales_Outlier"] = df.groupby("Sub_Category")["Sales"].transform(iqr_flag)
df["Profit_Outlier"] = df.groupby("Sub_Category")["Profit"].transform(iqr_flag)

df = df.sort_values("Order_Date").reset_index(drop=True)

# 8. Validation
assert df.isna().sum().sum() == 0
assert (df.Sales > 0).all() and (df.Quantity > 0).all()
assert np.allclose(df.Net_Profit, df.Gross_Profit - df.Operating_Expense)
print("All validation checks passed")

# 9. Save
df.to_csv(CLEAN_DIR / "clean_financial_data.csv", index=False)
df.to_excel(CLEAN_DIR / "clean_financial_data.xlsx", index=False)
print("Saved to:", CLEAN_DIR)
print("Clean shape:", df.shape)

Raw file: C:\Users\tanis\OneDrive\Desktop\cs-vision-internship\data\raw\Global_Superstore2(Orders).csv
Raw shape: (51290, 24)
All validation checks passed
Saved to: C:\Users\tanis\OneDrive\Desktop\cs-vision-internship\data\clean
Clean shape: (51290, 40)


**Key decisions:** duplicates removed (none found); Postal Code labelled "Unknown" (only US orders have one); negative-profit rows kept as real losses; outliers flagged, not removed; Cost = Sales − Profit; Net Profit = Profit − Shipping Cost.

## 2. Data Profiling Summary

In [6]:
print("Raw shape:", raw.shape)
raw.info()
m = raw.isna().sum()
print("\nMissing values in raw data:\n", m[m > 0])
print("Duplicate rows in raw data:", raw.duplicated().sum())
print("Date format example:", raw["Order Date"].head(3).tolist())
print("Negative-profit rows (kept):", (df.Profit < 0).sum())
print("Date range:", df.Order_Date.min().date(), "to", df.Order_Date.max().date())
print("Sales outliers flagged:", df.Sales_Outlier.sum(), "| Profit outliers flagged:", df.Profit_Outlier.sum())

Raw shape: (51290, 24)
<class 'pandas.DataFrame'>
RangeIndex: 51290 entries, 0 to 51289
Data columns (total 24 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Row ID          51290 non-null  int64  
 1   Order ID        51290 non-null  str    
 2   Order Date      51290 non-null  str    
 3   Ship Date       51290 non-null  str    
 4   Ship Mode       51290 non-null  str    
 5   Customer ID     51290 non-null  str    
 6   Customer Name   51290 non-null  str    
 7   Segment         51290 non-null  str    
 8   City            51290 non-null  str    
 9   State           51290 non-null  str    
 10  Country         51290 non-null  str    
 11  Postal Code     9994 non-null   float64
 12  Market          51290 non-null  str    
 13  Region          51290 non-null  str    
 14  Product ID      51290 non-null  str    
 15  Category        51290 non-null  str    
 16  Sub-Category    51290 non-null  str    
 17  Product Name    512

## 3. NumPy Analysis

In [7]:
sales  = df["Sales"].to_numpy()
profit = df["Profit"].to_numpy()
net    = df["Net_Profit"].to_numpy()
region = df["Region"].to_numpy()
disc   = df["Discount"].to_numpy()

print(f"Total sales: {sales.sum():,.2f}")
print(f"Total gross profit: {profit.sum():,.2f} | Total net profit: {net.sum():,.2f}")
print(f"Net margin: {net.sum()/sales.sum()*100:.2f}%")
print(f"Mean / median / std: {sales.mean():,.2f} / {np.median(sales):,.2f} / {sales.std(ddof=1):,.2f}")
print("Percentiles 25/50/75/95:", np.percentile(sales, [25, 50, 75, 95]).round(2))

loss = net < 0
print(f"\nLoss-making order lines: {loss.sum()} ({loss.mean()*100:.1f}%)")
print("Avg net profit, discount > 20%:", net[disc > 0.2].mean().round(2))
print("Avg net profit, no discount   :", net[disc == 0].mean().round(2))

print()
for r in np.unique(region):
    m = region == r
    print(f"{r:16s} sales={sales[m].sum():>12,.0f}  net profit={net[m].sum():>10,.0f}")

print("\nCorrelation Discount vs Profit:", np.corrcoef(disc, profit)[0, 1].round(3))

Total sales: 12,642,501.91
Total gross profit: 1,467,457.29 | Total net profit: 114,636.60
Net margin: 0.91%
Mean / median / std: 246.49 / 85.05 / 487.57
Percentiles 25/50/75/95: [  30.76   85.05  251.05 1015.96]

Loss-making order lines: 21150 (41.2%)
Avg net profit, discount > 20%: -89.92
Avg net profit, no discount   : 35.34

Africa           sales=     783,773  net profit=       732
Canada           sales=      66,928  net profit=    10,412
Caribbean        sales=     324,281  net profit=    -1,019
Central          sales=   2,822,303  net profit=    14,463
Central Asia     sales=     752,827  net profit=    54,650
EMEA             sales=     806,161  net profit=   -44,478
East             sales=     678,781  net profit=    18,779
North            sales=   1,248,166  net profit=    63,124
North Asia       sales=     848,310  net profit=    70,548
Oceania          sales=   1,100,185  net profit=      -539
South            sales=   1,600,907  net profit=   -28,118
Southeast Asia   sal

## 4. Verification of the Clean Dataset

In [8]:
import pandas as pd
df = pd.read_csv(r"C:\Users\tanis\OneDrive\Desktop\cs-vision-internship\data\clean\clean_financial_data.csv",
                 parse_dates=["Order_Date", "Ship_Date"])
print(df.shape)

(51290, 40)
